In [4]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [5]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


In [6]:
from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ import 완료


In [7]:
def run_with_tools(question: str, tools: list, llm_obj=llm):
    """질문 + 도구 리스트를 받아 4단계 흐름을 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


## 과제 C: 가상 날씨 API (실무 빈도 ★★★★☆)

In [8]:
class WeatherInfo(BaseModel):
    city: str = Field(description="도시")
    temperature_c: float = Field(description="도시의 온도")
    condition: Literal["맑음", "흐림", "비", "눈"] = Field(description="도시의 날씨")
    humidity_percent: int = Field(description="도시의 습도")

@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다.
    
    Args:
        city: 도시"""
    weathers = {"서울":{"temperature_c": 21.4, "condition": "맑음", "humidity_percent": 32},
                "부산":{"temperature_c": 23.1, "condition": "흐림", "humidity_percent": 46},
                "대전":{"temperature_c": 17.1, "condition": "비", "humidity_percent": 61},
                "울산":{"temperature_c": -3.2, "condition": "눈", "humidity_percent": 12}
                }
    weather = weathers.get(city,"")
    return WeatherInfo(city=city, temperature_c=weather["temperature_c"], condition=weather["condition"], humidity_percent=weather["humidity_percent"])

In [9]:
for q in ["서울이랑 부산 날씨 알려줘", "대전이랑 울산 날씨 어때?"]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_weather])}")
    print()

❓ 서울이랑 부산 날씨 알려줘
💬 현재 서울의 날씨는 다음과 같습니다:
- 온도: 21.4°C
- 날씨: 맑음
- 습도: 32%

부산의 날씨는 다음과 같습니다:
- 온도: 23.1°C
- 날씨: 흐림
- 습도: 46%

❓ 대전이랑 울산 날씨 어때?
💬 현재 대전의 날씨는 다음과 같습니다:
- 기온: 17.1°C
- 날씨: 비
- 습도: 61%

울산의 날씨는 다음과 같습니다:
- 기온: -3.2°C
- 날씨: 눈
- 습도: 12%



## 과제 D: docstring 일부러 짧게 써보기 (실험 ★★★★★)

In [ ]:
@tool
def get_weather_v1(city: str) -> WeatherInfo:
    """날씨"""
    weathers = {"서울":{"temperature_c": 21.4, "condition": "맑음", "humidity_percent": 32},
                "부산":{"temperature_c": 23.1, "condition": "흐림", "humidity_percent": 46},
                "대전":{"temperature_c": 17.1, "condition": "비", "humidity_percent": 61},
                "울산":{"temperature_c": -3.2, "condition": "눈", "humidity_percent": 12}
                }
    weather = weathers.get(city,"")
    return WeatherInfo(city=city, temperature_c=weather["temperature_c"], condition=weather["condition"], humidity_percent=weather["humidity_percent"])

@tool
def get_weather_v2(city: str) -> WeatherInfo:
    """도시의 현재 날씨와 온도 그리고 습도를 조회합니다.
    
    Args:
        city: 도시"""
    weathers = {"서울":{"temperature_c": 21.4, "condition": "맑음", "humidity_percent": 32},
                "부산":{"temperature_c": 23.1, "condition": "흐림", "humidity_percent": 46},
                "대전":{"temperature_c": 17.1, "condition": "비", "humidity_percent": 61},
                "울산":{"temperature_c": -3.2, "condition": "눈", "humidity_percent": 12}
                }
    weather = weathers.get(city,"")
    return WeatherInfo(city=city, temperature_c=weather["temperature_c"], condition=weather["condition"], humidity_percent=weather["humidity_percent"])

In [ ]:
for q in ["서울이랑 부산 날씨 알려줘", "대전이랑 울산 날씨 어때?"]:
    print("=========v1===========")
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_weather_v1])}")
for q in ["서울이랑 부산 날씨 알려줘", "대전이랑 울산 날씨 어때?"]:
    print("=========v2===========")
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_weather_v2])}")